# Python PIP

> 📘 **Python Mastery** · Module 05 — Intermediate Python · Lesson 6/7

The standard library is only the beginning. **pip** is Python's package manager: one command downloads and installs any of the million-plus packages on PyPI, built by the community.

## 🎯 Learning Objectives

- Explain what pip and PyPI are
- Install, upgrade and remove packages from the terminal
- Inspect what is installed with `pip list` and `pip show`
- Locate where packages physically live on disk (site-packages)
- Freeze an environment into `requirements.txt` and rebuild it elsewhere
- Read version specifiers (`>=`, `<`, `~=`) and judge a package's quality

## 1. What Are pip and PyPI?

**PyPI** -- the Python Package Index (<https://pypi.org>) -- is a public warehouse of installable packages: `requests` for HTTP, `numpy` for arrays, `pandas` for tables, `pygame` for games...

**pip** is the delivery service. It reads a package name, finds it on PyPI, resolves everything that package depends on, and installs the whole stack into your Python.

> 🔍 **Under the Hood:** `pip install` does more than copy files. It fetches a wheel, checks hashes against PyPI's metadata, solves the dependency graph for mutually compatible versions, unpacks into `site-packages`, and registers scripts/entry points. Each install leaves a `*.dist-info` folder behind -- that registry is exactly what `pip list` and `pip show` read.

If you know JavaScript: PyPI is npm's registry, pip is `npm install`. If you know phones: PyPI is the app store, pip is the install button.

> All commands in this lesson are **terminal commands** -- run them in your shell (PowerShell, cmd, Git Bash), never inside Python or a notebook code cell.

## 2. Checking Your pip Version

Before anything else, confirm pip exists and which Python it belongs to. The version line conveniently prints its path.

```bash
pip --version
```

Preferred, unambiguous form -- uses *exactly* the Python you invoked:

```bash
python -m pip --version
```

Typical output:

```text
pip 25.2 from C:\Users\fahim\venv\Lib\site-packages\pip (python 3.14)
```

If several Pythons share a machine, `python -m pip` guarantees the packages land in the interpreter you are actually using.

## 3. Installing Packages

One command, latest stable version:

```bash
pip install requests
```

Pin to an exact version (reproducibility!):

```bash
pip install requests==2.32.3
```

Or constrain a range -- at least 2.31 but below 3.0:

```bash
pip install "requests>=2.31,<3"
```

Install several at once:

```bash
pip install numpy pandas matplotlib
```

Quotes matter on Windows PowerShell and Linux shells alike -- `<` and `>` would otherwise be read as redirection. Make a habit of quoting anything containing comparison signs.

## 4. Upgrading and Uninstalling

Packages evolve; keep them current or jump versions deliberately:

```bash
pip install -U requests              # -U = --upgrade, to the newest allowed
pip install requests==2.32.3         # "downgrade" by pinning an older version
pip uninstall requests               # asks for confirmation
pip uninstall -y requests            # skips the prompt (handy in scripts)
```

Note there is no dedicated `update-all`; careful projects upgrade deliberately, one package at a time, then re-run their tests.

## 5. Listing and Inspecting Installed Packages

What do I have, and what exactly is it?

```bash
pip list                    # every installed package + version
pip list --outdated         # which ones have newer releases
pip show numpy              # version, author, homepage, dependencies of ONE package
```

`pip show` is the first tool to reach for when debugging "works on my machine" reports: it tells you precisely what is installed *here*.

## 6. Where Do Packages Land? site-packages

Every installed package unpacks into a folder called **site-packages**, inside your Python installation (or virtual environment -- next lesson). The `site` module knows the address:

**Example:** ask Python itself where third-party code goes.

In [ ]:
import site
import sys

print("packages land in:", site.getsitepackages()[0])
print("this interpreter :", sys.executable)
print("already visible  :", site.ENABLE_USER_SITE)

In [ ]:
from importlib.metadata import version, metadata

for pkg in ["numpy", "pandas", "matplotlib"]:
    try:
        print(f"{pkg:<12} {version(pkg)}")
    except Exception:
        print(f"{pkg:<12} not installed in this environment")

info = metadata("pip")
print("even pip itself:", info["Name"], info["Version"])

## 7. requirements.txt: The Environment Recipe

Your project needs numpy 2.x, requests 2.32... and so does your teammate's laptop, and the production server. Instead of emailing a list of commands, capture the environment in one plain-text file named `requirements.txt`:

```text
numpy==2.3.1
pandas>=2.2,<3
requests~=2.32
matplotlib
```

Generate it from your current environment:

```bash
pip freeze > requirements.txt
```

Rebuild the identical environment anywhere:

```bash
pip install -r requirements.txt
```

Commit this file to git; never commit the installed packages themselves. It turns "it works on my machine" into "here is the machine, in eleven lines".

## 8. Version Specifiers Cheat Sheet

| Specifier | Meaning | Example |
|---|---|---|
| `==2.3.1` | exactly this version | hard pin -- maximum reproducibility |
| `>=2.2` | this version or newer | minimum floor |
| `<3` | strictly below | avoid a breaking major release |
| `>=2.2,<3` | floor AND ceiling together | the common sweet spot |
| `~=2.32` | `>=2.32, <3.0` -- compatible within the major | accepts 2.33, rejects 3.0 |
| `!=1.9` | everything except this buggy release | escape hatch |
| bare `matplotlib` | any version -- latest at install time | fine for toys, risky for teams |

Rule of thumb: applications pin tightly (`==`); libraries stay loose (`>=,<`) so they can coexist with others.

## 9. Choosing Good Packages

Anyone can upload to PyPI, so a quick vetting ritual saves painful nights:

- **Popularity** -- downloads per month and stars signal battle-testing.
- **Maintenance** -- recent release? issues answered? Or abandoned since 2019?
- **Dependencies** -- a tiny utility dragging in 40 packages is a liability chain.
- **License** -- MIT/Apache/BSD are permissive; GPL obligations differ; check before shipping.
- **Docs & types** -- good README, type hints and changelogs predict pleasant development.

Security note: typosquatting is real -- `requestz` is not `requests`. Copy exact names from official docs or pypi.org, and prefer `pip install` over copy-pasting random `curl | bash` instructions.

## 10. Installing from Other Sources

PyPI is the default source, but pip can install from anywhere:

```bash
pip install ./downloads/numpy-2.3.1-cp314.whl        # a local wheel file
pip install git+https://github.com/user/repo.git      # straight from GitHub
pip install requests --no-cache-dir                   # bypass the local cache
```

You'll meet this when testing an unreleased fix or installing an internal company package. For everyday work, plain PyPI names are all you need.

## 11. Your First Install: A Guided Tour

A complete round trip you can run today in your own terminal:

```bash
python -m pip --version                       # 1. who am I using?
python -m pip install rich                    # 2. fetch a pretty-terminal library
python -m pip show rich                       # 3. inspect it
python -m rich                                # 4. its built-in demo
python -m pip freeze > requirements.txt       # 5. snapshot the environment
type requirements.txt                         #    (Windows) peek inside
python -m pip uninstall -y rich               # 6. clean up
```

Notice every step uses `python -m pip` -- the habit that keeps installs attached to the interpreter you intend.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Running bare `pip` with several Pythons installed | package lands in a different interpreter than the one you run | always `python -m pip ...` |
| Installing everything globally | version soup; upgrading one project breaks another | use virtual environments (Lesson 7) |
| Committing the venv folder instead of requirements.txt | megabytes of machine-specific binaries in git | commit the recipe, not the kitchen |
| Unpinned requirements ("whatever was latest") | teammates get different versions; builds stop reproducing | `pip freeze > requirements.txt` |
| Typing a package name from memory | typosquat packages impersonate popular names | copy exact spelling from official docs/pypi.org |
| Forgetting quotes around `"pkg>=2,<3"` | shell interprets `<` as file redirection | quote specifiers containing `<` or `>` |

## 💡 Best Practices & Pro Tips

- Adopt the reflex `python -m pip` -- it can never target the wrong interpreter.
- Regenerate `requirements.txt` after every deliberate dependency change, and review it like code.
- Keep pip itself fresh inside each environment: `python -m pip install -U pip`.
- `pip cache dir` / `pip cache purge` -- reinstalls are cached, so disk bloat is rare but cleanable.
- **AI-engineering relevance:** ML stacks pin aggressively because CUDA/torch/transformer versions interlock; Dockerfiles install from frozen requirements so training runs are byte-for-byte reproducible months later.

## 📌 Summary

| Command | What it does |
|---|---|
| `python -m pip --version` | confirm pip and its Python |
| `pip install <pkg>` | install latest from PyPI |
| `pip install <pkg>==2.32.3` | install an exact version |
| `pip install -U <pkg>` | upgrade to newest allowed |
| `pip uninstall <pkg>` | remove a package |
| `pip list` / `pip show <pkg>` | inventory / deep detail |
| `pip freeze > requirements.txt` | snapshot the environment |
| `pip install -r requirements.txt` | rebuild from the snapshot |

Key takeaways:

- PyPI is the warehouse, pip the courier; both come bundled with modern Python.
- `requirements.txt` converts an environment into a portable recipe -- commit it.
- Pin deliberately: apps want `==`, libraries want ranges.
- Vetting popularity, maintenance and license is part of installing responsibly.

## 🔗 Next Lesson

Up next: **[07_Virtualenv](../07_Virtualenv/notes.ipynb)** -- giving every project its own private Python so package conflicts disappear for good.